### 문제 2-1 : 콤마 구분 리스트 파서 활용



#### 문제 설명
사용자가 관심있는 분야(예: "음식", "스포츠", "영화" 등)를 입력하면, 해당 분야와 관련된 한국의 유명한 장소나 활동 5가지를 콤마로 구분된 리스트로 출력하는 프로그램을 작성하세요.

출력파셔 : CommaSeparatedListOutputParser

요구사항
- CommaSeparatedListOutputParser 사용
- 사용자 입력을 받아 동적으로 처리
- 한국 관련 내용으로 추천


In [ ]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

In [ ]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-5.6-luna",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None, verbose: bool = True, **kwargs) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다.
    kwargs: ChatOpenAI 에 그대로 전달할 추가 옵션 (예: timeout=60)"""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
        **kwargs,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [ ]:
#파서 만들기
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import csv

output_parser = CommaSeparatedListOutputParser()
format_instructions = output_parser.get_format_instructions()
print(format_instructions)


In [ ]:
#프롬프트 만들기
prompt = PromptTemplate(
    template="""{subject}분야와 관련된 한국의 유명한 장소 또는 활동 5가지 추천.\n{format_instructions}""",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions},
)

llm = get_llm(temperature=0.7)

#프롬프트 문장 확인.
print(prompt.format(subject="영화"))

In [ ]:
#체인 연결
chain = prompt | llm | output_parser

In [ ]:
#사용자 입력받아 동적 처리
# 실행
user_recommend = input("주제를 입력하면 5가지를 추천해드립니다👍: ")
result = chain.invoke({"subject": user_recommend})

print(f"\n '{user_recommend}' 관련 한국의 유명한 장소나 활동 5가지 추천 ")
print(result)


### 문제 2-2 : 영화 리뷰 감정 분석기


#### 문제 설명
영화 리뷰 텍스트를 입력받아 감정을 "긍정", "부정", "보통" 중 하나로 분류하는 시스템을 만드세요.

출력파셔 : EnumOutputParser

요구사항
- EnumOutputParser와 Enum 클래스 사용
- 여러 개의 테스트 리뷰로 검증
- 감정 분류 결과를 깔끔하게 출력

테스트 리뷰 예시:
"이 영화 정말 재미없어요. 시간 낭비였습니다."
"배우들의 연기가 훌륭하고 스토리도 감동적이었어요!"
"그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요."

분석실패했을 경우,
OutputFixingParser 와 EnumOutputParser 를 같이 사용(선택)



In [ ]:
from enum import Enum
from langchain_classic.output_parsers import EnumOutputParser  # langchain 1.x: classic 패키지에서 import
from langchain_core.prompts import ChatPromptTemplate

class Sentiment(Enum):
    """리뷰 감성 분류 값"""
    POSITIVE = "긍정"
    NEGATIVE = "부정"
    NEUTRAL = "중립"

# Enum 출력 파서 초기화
parser = EnumOutputParser(enum=Sentiment)

# 출력 형식 지침 확인: "Select one of the following options: 긍정, 부정, 중립"
format_instructions = parser.get_format_instructions()
print(format_instructions)

# 모델 없이 파서만 테스트
print(repr(parser.parse("긍정")))
print(repr(parser.parse(" 부정\n")))
#sentiment_parser.parse("좋음")

In [ ]:
# 프롬프트 템플릿 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 리뷰 감성 분류 전문가입니다. 리뷰를 읽고 감성을 분류하세요. 설명 없이 값만 답하세요.\n{format_instructions}"),
    ("user", "리뷰: {review}"),
]).partial(format_instructions=format_instructions)

# LLM 초기화 (분류 작업이므로 temperature 를 0으로 설정)
llm = get_llm(temperature=0)

# 체인 생성
chain = prompt | llm | parser

In [ ]:
from langchain_core.exceptions import OutputParserException

reviews = [
    "이 영화 정말 재미없어요. 시간 낭비였습니다.",
    "배우들의 연기가 훌륭하고 스토리도 감동적이었어요!",
    "그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요.",
]

for review in reviews:
    print(f"리뷰: {review}")
    try:
        result = chain.invoke({"review": review})
        # 결과는 문자열이 아니라 Sentiment Enum 멤버
        print(f"  분류 결과: {result} / 타입: {type(result).__name__} / 값: {result.value}")
    except OutputParserException as e:
        # 모델이 Enum 에 없는 값으로 답하면 파싱 오류가 발생
        print(f"  파싱 오류: {e}")
    print("-" * 50)

#### EnumOutputParser + OutputFixingParser (파싱 오류 자동 복구)


In [ ]:
from langchain_classic.output_parsers import OutputFixingParser
from langchain_core.exceptions import OutputParserException

fixing_parser = OutputFixingParser.from_llm(llm=llm, parser=parser, max_retries=1)

bad_completion = "Null Value"

try:
    parser.parse(bad_completion)
except OutputParserException as e:
    print("기본 파서 오류:", str(e).splitlines()[0])

fixed = fixing_parser.parse(bad_completion)
print(f"수정 후 결과: {fixed} / 값: {fixed.value}")

In [ ]:
# 체인에 OutputFixingParser 연결: 프롬프트 → 모델 → 오류 자동 복구 파서
fixing_chain = prompt | llm | fixing_parser

for review in reviews:
    print(f"리뷰: {review}")
    try:
        result = fixing_chain.invoke({"review": review})
        print(f"  분류 결과: {result} / 값: {result.value}")
    except OutputParserException as e:
        # 수정 시도 후에도 Enum 값이 아니면 오류 발생
        print(f"  파싱 오류(수정 실패): {str(e).splitlines()[0]}")
    print("-" * 50)

### 문제 2-3 : 학생 정보 구조화 시스템


#### 문제 설명
학생의 자유 형식 자기소개를 입력받아 이름, 나이, 전공, 취미 리스트, 목표를 구조화된 형태로 추출하는 시스템을 만드세요.

출력파셔 : PydanticOutputParser

요구사항
- PydanticOutputParser와 BaseModel 사용
- 각 필드에 적절한 타입과 설명 추가
- 자유 형식의 텍스트에서 정보 추출


예시 입력:
 "안녕하세요! 저는 김민수이고 22살입니다. 컴퓨터공학을 전공하고 있어요. 취미로는 게임하기, 영화보기, 코딩을 좋아합니다. 앞으로 훌륭한 개발자가 되는 것이 목표입니다."

예시 출력 구조:
{
    "name": "김민수",
    "age": 22,
    "major": "컴퓨터공학",
    "hobbies": ["게임하기", "영화보기", "코딩"],
    "goal": "훌륭한 개발자가 되는 것"
}


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

from pydantic import BaseModel, Field
from typing import List
from pprint import pprint

class StudentInfo(BaseModel):
    name: str = Field(description="학생 이름")
    age: int = Field(description="학생 나이 (숫자만)")
    major: str = Field(description="전공")
    hobbies: List[str] = Field(description="취미 목록")
    goal: str = Field(description="앞으로의 목표")


In [ ]:
parser = PydanticOutputParser(pydantic_object=StudentInfo)
print(parser.get_format_instructions())

In [ ]:
template = """
다음 학생의 자기소개에서 정보를 추출해주세요.
자기소개: {query}

{format_instructions}
"""

prompt = ChatPromptTemplate.from_template(template)

prompt = prompt.partial(
    format_instructions=parser.get_format_instructions()
)
pprint(prompt)

model = get_llm(temperature=0.5)

In [ ]:
query = "안녕하세요! 저는 김민수이고 22살입니다. 컴퓨터공학을 전공하고 있어요. 취미로는 게임하기, 영화보기, 코딩을 좋아합니다. 앞으로 훌륭한 개발자가 되는 것이 목표입니다."
chain = prompt | model | parser
output = chain.invoke({"query": query})

print(type(output))
pprint(output)

In [ ]:
print(f"이름: {output.name}")
print(f"나이: {output.age}")
print(f"전공: {output.major}")
print(f"취미: {', '.join(output.hobbies)}")  # List => str 로 변환
print(f"목표: {output.goal}")

In [ ]:
import json
print(json.dumps(output.model_dump(), ensure_ascii=False, indent=4))

### 문제 2-4 : 여행 계획 분석기

출력 파서: StructuredOutputParser

문제 :
여행 후기나 계획 텍스트를 입력받아 여행지, 기간, 예산, 추천도(1-5점), 주요 활동 리스트를 구조화된 형태로 추출하는 시스템을 만드세요.

요구사항:
- StructuredOutputParser와 ResponseSchema 사용
- 5개의 필드를 정의하여 정보 추출
- 자연어 텍스트에서 핵심 정보 파싱

예시 입력:
 "지난 주에 부산으로 2박 3일 여행을 다녀왔어요. 총 30만원 정도 썼는데 해운대에서 바다구경하고, 자갈치시장에서 회 먹고, 감천문화마을도 구경했어요. 정말 만족스러운 여행이었습니다. 5점 만점에 4점 정도 줄 수 있을 것 같아요."

예시 출력 구조:
{
    "destination": "부산",
    "duration": "2박 3일",
    "budget": "30만원",
    "rating": "4",
    "activities": ["해운대 바다구경", "자갈치시장 회 먹기", "감천문화마을 구경"]
}


In [ ]:
import json
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema  # langchain 1.x: classic 패키지에서 import
from langchain_core.prompts import ChatPromptTemplate

response_schemas = [
    ResponseSchema(name="destination", description="여행지 (예: 부산)"),
    ResponseSchema(name="duration", description="여행 기간 (예: 2박 3일)"),
    ResponseSchema(name="budget", description="총 예산 (예: 30만원)"),
    ResponseSchema(name="rating", description="추천도, 1~5 사이 숫자 하나 (예: 4)"),
    ResponseSchema(name="activities", description="주요 활동 목록", type="List[string]"),
]

parser = StructuredOutputParser.from_response_schemas(response_schemas)

format_instructions = parser.get_format_instructions()
print(format_instructions)

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 후기 분석 전문가입니다. 여행 후기에서 정보를 추출하고 반드시 아래 형식을 지키세요.\n{format_instructions}"),
    ("user", "{question}"),
]).partial(format_instructions=format_instructions)

llm = get_llm(temperature=0)

chain = prompt | llm | parser

In [ ]:
from langchain_core.exceptions import OutputParserException

question = "지난 주에 부산으로 2박 3일 여행을 다녀왔어요. 총 30만원 정도 썼는데 해운대에서 바다구경하고, 자갈치시장에서 회 먹고, 감천문화마을도 구경했어요. 정말 만족스러운 여행이었습니다. 5점 만점에 4점 정도 줄 수 있을 것 같아요."

try:
    response = chain.invoke({"question": question})
    print(type(response))  # 결과는 dict

   
    print(json.dumps(response, indent=4, ensure_ascii=False))

    
    print("여행지:", response["destination"])
    print("기간:", response["duration"])
    print("예산:", response["budget"])
    print("추천도:", response["rating"])
    print("주요 활동:", ", ".join(response["activities"]))
except OutputParserException as e:
    
    print(f"파싱 오류: {str(e).splitlines()[0]}")